# Enhancement model exploration: additive vs multiplicative `S`

We want to model the contrast enhancement that turns T1 into T1CE. Two candidate models,
restricted to the **enhancing tumour (ET)** region where enhancement actually happens:

| model | form | "supervised" target |
|---|---|---|
| additive | `T1CE = T1 + S` | `S = T1CE - T1` |
| multiplicative | `T1CE = T1 * S` | `S = T1CE / T1` |

This notebook does three things:

1. **Regression of ET pixels** — fit both models (plus a free affine reference) and decide
   which is better supported.
2. **Raw vs z-scored data** — the `_img.h5` stores both; the choice of space changes which
   model even *makes sense*.
3. **Example "supervised" `S`** computed from the scaled T1 / T1CE the loaders actually deliver.

The ET mask is **not** in the `_img.h5` — we load `*_seg.nii.gz` from the subject folder and
replay the exact crop + slice selection `preprocessing/cmap.py` applied, so it lines up.

## 0. Setup

In [ ]:
import os, sys, glob
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import h5py
import nibabel as nib
from preprocessing.cmap import center_crop_spatial     # reuse the EXACT crop cmap.py used

# ---- point this at a split (or the full dataset) root: one folder per subject ----
DATA_ROOT  = "/home/ee2178/scratch/ee2178/datasets/BraTS/BraTS2021_DataSet_train"
RAW_ROOT   = None          # fallback root holding *_seg.nii.gz, if not next to the h5
N_SUBJECTS = 8             # subjects to pool for the regression
ET_LABEL   = 4             # BraTS2021 seg: 1=NCR/NET, 2=ED, 4=ET
T1_IDX, T1CE_IDX = 1, 2    # stored order [flair, t1, t1ce, t2]
RNG        = np.random.default_rng(0)
# (the intensity scale for section 4 is CHOSEN there, off the raw histogram)

print("repo:", REPO_ROOT)
print("data:", DATA_ROOT)

## 1. Loading + aligning the ET mask

`cmap.py` wrote each `_img.h5` after (a) a center crop to `crop_size` and (b) keeping only a
subset of slices, recorded in `slice_index` (indices into the original nifti z-axis). To align
the segmentation we replay both steps in the same order. `nib` gives `(H, W, D)`; cmap
permuted to `(D, H, W)`, so we transpose first.

In [ ]:
# Locate *_seg.nii.gz for this subject (works through the split symlinks).
def find_seg(img_h5):
    d = os.path.dirname(os.path.realpath(img_h5))
    hits = glob.glob(os.path.join(d, "*_seg.nii.gz"))
    if hits:
        return hits[0]
    if RAW_ROOT:                                   # fallback: RAW_ROOT/<case>/<case>_seg.nii.gz
        case = os.path.basename(img_h5).replace("_img.h5", "")
        hits = glob.glob(os.path.join(RAW_ROOT, case, "*_seg.nii.gz"))
        if hits:
            return hits[0]
    return None


# Return dict with img (z-scored), img_raw, brain mask, ET mask, attrs -- all aligned
# (N, H, W, ...). ET is None if the segmentation could not be found.
def load_subject(img_h5, want_seg=True):
    with h5py.File(img_h5, "r") as f:
        out = dict(img=np.asarray(f["img"][:]), attrs=dict(f.attrs),
                   mask=np.asarray(f["mask"][:])[..., 0].astype(bool),
                   slice_index=np.asarray(f["slice_index"][:]))
        out["img_raw"] = np.asarray(f["img_raw"][:]) if "img_raw" in f else None
    names = out["attrs"].get("contrasts", "")
    out["names"] = names.split(",") if isinstance(names, str) else [str(c) for c in names]
    out["crop"] = int(out["attrs"].get("crop_size", -1))

    out["et"] = out["seg"] = None
    if want_seg:
        seg_path = find_seg(img_h5)
        if seg_path is None:
            return out
        seg = np.asarray(nib.load(seg_path).get_fdata())        # (H, W, D)
        seg = np.transpose(seg, (2, 0, 1))                      # (D, H, W)  -- cmap's orientation
        if out["crop"] and out["crop"] > 0:                     # SAME center crop as cmap.py
            seg = center_crop_spatial(seg, out["crop"], h_axis=1, w_axis=2)
        seg = seg[out["slice_index"]]                           # SAME kept slices
        out["seg"] = np.rint(seg).astype(np.int16)
        out["et"] = out["seg"] == ET_LABEL
        out["seg_path"] = seg_path
    return out


subjects = sorted(glob.glob(os.path.join(DATA_ROOT, "*", "*_img.h5")))
print(f"found {len(subjects)} subject h5 files")
assert subjects, f"no *_img.h5 under {DATA_ROOT}"

s = load_subject(subjects[0])
print("img      :", s["img"].shape, "| img_raw:", None if s["img_raw"] is None else s["img_raw"].shape)
print("contrasts:", s["names"], "| crop_size:", s["crop"], "| normalize:", s["attrs"].get("normalize"))
if s["et"] is None:
    print("\n!! segmentation NOT found -- set RAW_ROOT to the folder holding <case>/<case>_seg.nii.gz")
else:
    print("seg      :", s["seg"].shape, "labels:", np.unique(s["seg"]), "<-", os.path.basename(s["seg_path"]))
    print(f"ET voxels: {s['et'].sum()}  ({100 * s['et'].mean():.3f}% of FOV)")

### Alignment sanity check
ET must sit **inside** the brain mask and land on bright T1CE.

In [ ]:
assert s["seg"].shape == s["img"].shape[:3], (s["seg"].shape, s["img"].shape)
inside = (s["et"] & s["mask"]).sum() / max(s["et"].sum(), 1)
print(f"fraction of ET voxels inside the brain mask: {inside:.4f}   (should be ~1.0)")

t1, t1ce = s["img"][..., T1_IDX], s["img"][..., T1CE_IDX]
print(f"mean z-scored T1CE  in ET: {t1ce[s['et']].mean():+.3f}   vs whole brain: {t1ce[s['mask']].mean():+.3f}")
print(f"mean z-scored T1    in ET: {t1[s['et']].mean():+.3f}   vs whole brain: {t1[s['mask']].mean():+.3f}")
print("(ET should be clearly BRIGHTER than brain average in T1CE -- that is the enhancement)")

# visual: the slice with the most ET
zbest = int(s["et"].reshape(s["et"].shape[0], -1).sum(1).argmax())
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (ttl, im) in zip(ax, [("T1", t1[zbest]), ("T1CE", t1ce[zbest]),
                             ("T1CE - T1", t1ce[zbest] - t1[zbest]), ("seg", s["seg"][zbest])]):
    if ttl == "seg":
        a.imshow(s["seg"][zbest], cmap="viridis", interpolation="nearest")
    else:
        a.imshow(im, cmap="gray", vmin=-2, vmax=3)
        a.contour(s["et"][zbest], levels=[0.5], colors="r", linewidths=0.8)   # ET outline
    a.set_title(f"{ttl}  (z={zbest})"); a.axis("off")
fig.suptitle("ET outline (red) overlaid -- it must hug the bright T1CE rim", y=1.02)
plt.tight_layout(); plt.show()

## 2. Regression of ET pixels: additive vs multiplicative

Three least-squares fits of `T1CE` on `T1`, using ET voxels only:

| fit | form | free params |
|---|---|---|
| additive | `T1CE = T1 + b` | 1 (`b = S`) |
| multiplicative | `T1CE = a * T1` | 1 (`a = S`) |
| affine (reference) | `T1CE = a*T1 + b` | 2 |

The two candidates have the **same number of parameters**, so RMSE / R² compare them fairly.
The affine fit is the upper bound and tells us *why*: `a ~ 1` favours additive, `b ~ 0`
favours multiplicative.

We also check whether the implied `S` is **independent of `T1`** — the real assumption behind
each model. If `S = T1CE - T1` still correlates with `T1`, the additive model is leaving
structure on the table (and vice versa).

In [ ]:
# Least squares for y = x + b, y = a*x, and y = a*x + b. Returns a metrics dict.
def fit_models(x, y):
    x = x.astype(np.float64); y = y.astype(np.float64)
    sst = ((y - y.mean()) ** 2).sum()
    r2 = lambda sse: 1.0 - sse / sst if sst > 0 else np.nan
    rmse = lambda sse: np.sqrt(sse / x.size)

    b_add = (y - x).mean()                                   # additive: slope fixed at 1
    sse_add = ((y - x - b_add) ** 2).sum()

    a_mul = (x * y).sum() / max((x * x).sum(), 1e-12)        # multiplicative: intercept fixed at 0
    sse_mul = ((y - a_mul * x) ** 2).sum()

    a_aff, b_aff = np.polyfit(x, y, 1)                       # free affine reference
    sse_aff = ((y - a_aff * x - b_aff) ** 2).sum()

    with np.errstate(divide="ignore", invalid="ignore"):
        s_mul = y / x
    s_add = y - x
    ok = np.isfinite(s_mul)
    corr = lambda u, v: float(np.corrcoef(u, v)[0, 1]) if u.size > 2 and u.std() > 0 else np.nan
    cv = lambda v: float(np.std(v) / abs(np.mean(v))) if abs(np.mean(v)) > 1e-12 else np.inf

    return dict(n=x.size,
                b_add=b_add, rmse_add=rmse(sse_add), r2_add=r2(sse_add),
                a_mul=a_mul, rmse_mul=rmse(sse_mul), r2_mul=r2(sse_mul),
                a_aff=a_aff, b_aff=b_aff, rmse_aff=rmse(sse_aff), r2_aff=r2(sse_aff),
                corr_Sadd_T1=corr(s_add, x), corr_Smul_T1=corr(s_mul[ok], x[ok]),
                cv_Sadd=cv(s_add), cv_Smul=cv(s_mul[ok]))


# Pool (T1, T1CE) voxel pairs over subjects from `space`, restricted to `region`.
def collect(subject_paths, space="img_raw", region="et"):
    xs, ys, per_subject = [], [], []
    for p in subject_paths:
        sub = load_subject(p)
        arr = sub[space]
        if arr is None or sub["et"] is None:
            continue
        sel = sub["et"] if region == "et" else sub["mask"]
        if sel.sum() < 100:
            continue
        x, y = arr[..., T1_IDX][sel], arr[..., T1CE_IDX][sel]
        xs.append(x); ys.append(y)
        per_subject.append((os.path.basename(p).replace("_img.h5", ""), fit_models(x, y)))
    return np.concatenate(xs), np.concatenate(ys), per_subject


paths = subjects[:N_SUBJECTS]
HAS_RAW = load_subject(paths[0], want_seg=False)["img_raw"] is not None
SPACE = "img_raw" if HAS_RAW else "img"
print(f"pooling {len(paths)} subjects | space = {SPACE}"
      + ("" if HAS_RAW else "   (!! no img_raw -- regenerate with save_raw_image: true)"))

x_raw, y_raw, per_sub = collect(paths, space=SPACE, region="et")
res_raw = fit_models(x_raw, y_raw)
print(f"pooled ET voxels: {res_raw['n']:,}\n")

def report(r, title):
    print(f"--- {title} ---")
    print(f"  additive        T1CE = T1 + {r['b_add']:+.4f}          RMSE={r['rmse_add']:.4f}  R2={r['r2_add']:+.4f}")
    print(f"  multiplicative  T1CE = {r['a_mul']:.4f} * T1           RMSE={r['rmse_mul']:.4f}  R2={r['r2_mul']:+.4f}")
    print(f"  affine (ref)    T1CE = {r['a_aff']:.4f}*T1 {r['b_aff']:+.4f}   RMSE={r['rmse_aff']:.4f}  R2={r['r2_aff']:+.4f}")
    print(f"  corr(S,T1):  additive {r['corr_Sadd_T1']:+.3f}   multiplicative {r['corr_Smul_T1']:+.3f}   (0 = model assumption holds)")
    print(f"  CV(S)     :  additive {r['cv_Sadd']:.3f}        multiplicative {r['cv_Smul']:.3f}        (lower = more constant S)")
    better = "ADDITIVE" if r["rmse_add"] < r["rmse_mul"] else "MULTIPLICATIVE"
    print(f"  ==> lower RMSE: {better}")

report(res_raw, f"pooled ET voxels, {SPACE}")

### The scatter that decides it

In [ ]:
n_show = min(40000, x_raw.size)
idx = RNG.choice(x_raw.size, n_show, replace=False)
xs_, ys_ = x_raw[idx], y_raw[idx]
grid = np.linspace(np.percentile(x_raw, 0.5), np.percentile(x_raw, 99.5), 100)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].hexbin(xs_, ys_, gridsize=60, bins="log", cmap="viridis")
ax[0].plot(grid, grid + res_raw["b_add"], "r-",  lw=2, label=f"additive  (+{res_raw['b_add']:.3f})")
ax[0].plot(grid, res_raw["a_mul"] * grid, "c--", lw=2, label=f"multiplicative  (x{res_raw['a_mul']:.3f})")
ax[0].plot(grid, res_raw["a_aff"] * grid + res_raw["b_aff"], "y:", lw=2,
           label=f"affine  ({res_raw['a_aff']:.2f}x{res_raw['b_aff']:+.2f})")
ax[0].plot(grid, grid, color="w", lw=0.8, alpha=0.6, label="identity")
ax[0].set_xlabel("T1 (ET voxels)"); ax[0].set_ylabel("T1CE"); ax[0].legend(fontsize=8)
ax[0].set_title(f"ET voxels, {SPACE}: which model fits?")

# does S still depend on T1?
with np.errstate(divide="ignore", invalid="ignore"):
    s_add, s_mul = ys_ - xs_, ys_ / xs_
ax[1].plot(xs_, s_add, ".", ms=1, alpha=0.25, label=f"S_add = T1CE-T1  (r={res_raw['corr_Sadd_T1']:+.2f})")
ax[1].axhline(res_raw["b_add"], color="r", lw=1.5)
ax[1].set_xlabel("T1"); ax[1].set_ylabel("S_add"); ax[1].legend(fontsize=8)
ax[1].set_title("Additive S vs T1  (flat cloud => additive assumption OK)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(s_add, bins=150); ax[0].axvline(res_raw["b_add"], color="r")
ax[0].set_title(f"S_add = T1CE - T1   (mean {res_raw['b_add']:.3f}, CV {res_raw['cv_Sadd']:.2f})")
fin = np.isfinite(s_mul)
lo, hi = np.percentile(s_mul[fin], [1, 99])
ax[1].hist(s_mul[fin], bins=150, range=(lo, hi)); ax[1].axvline(res_raw["a_mul"], color="c")
ax[1].set_title(f"S_mul = T1CE / T1   (fit {res_raw['a_mul']:.3f}, CV {res_raw['cv_Smul']:.2f})")
for a in ax: a.set_xlabel("S"); a.set_ylabel("count")
plt.tight_layout(); plt.show()

### Per-subject consistency
A good model should give a **stable** `S` across subjects, not just on average.

In [ ]:
print(f"{'subject':<20}{'n_ET':>9}{'b_add':>9}{'a_mul':>9}{'RMSE_add':>10}{'RMSE_mul':>10}  winner")
for name, r in per_sub:
    win = "add" if r["rmse_add"] < r["rmse_mul"] else "mul"
    print(f"{name:<20}{r['n']:>9,}{r['b_add']:>9.3f}{r['a_mul']:>9.3f}"
          f"{r['rmse_add']:>10.3f}{r['rmse_mul']:>10.3f}  {win}")

wins_add = sum(r["rmse_add"] < r["rmse_mul"] for _, r in per_sub)
print(f"\nadditive wins on {wins_add}/{len(per_sub)} subjects")
b = np.array([r["b_add"] for _, r in per_sub]); a = np.array([r["a_mul"] for _, r in per_sub])
print(f"b_add across subjects: mean {b.mean():.3f}  std {b.std():.3f}  (spread/mean {abs(b.std()/b.mean()):.2f})")
print(f"a_mul across subjects: mean {a.mean():.3f}  std {a.std():.3f}  (spread/mean {abs(a.std()/a.mean()):.2f})")

## 3. Raw vs z-scored data

`_img.h5` holds both `img_raw` (unnormalized, background 0) and `img` (percentile-clipped, then
**per-channel** z-scored within the brain). `norm_stats` stores each channel's `(mean, std)`.

This matters for the model choice:

* **Additive** `S` is preserved by a *shared* affine map, but T1 and T1CE are z-scored with
  **different** `(mu, sigma)`, so `S_add` in z-space is *not* a rescaled `S_add` in raw space.
* **Multiplicative** `S` is scale-covariant but **not shift-invariant**. Z-scoring subtracts a
  mean, so z-scored T1 crosses zero *inside the brain* — the ratio `T1CE/T1` then blows up.
  That is exactly why a quotient histogram on z-scored data has enormous tails.

In [ ]:
stats = np.asarray(s["attrs"]["norm_stats"])          # (C, 2) = (mean, std) per channel
print(f"{'channel':<8}{'raw mean':>11}{'raw std':>10}{'raw min':>10}{'raw max':>10}"
      f"{'  |  z mean':>12}{'z std':>9}{'z min':>9}{'z max':>9}{'   norm_stats (mu, sd)':>26}")
for c, nm in enumerate(s["names"]):
    zb = s["img"][..., c][s["mask"]]
    line = f"{nm:<8}"
    if s["img_raw"] is not None:
        rb = s["img_raw"][..., c][s["mask"]]
        line += f"{rb.mean():>11.1f}{rb.std():>10.1f}{rb.min():>10.1f}{rb.max():>10.1f}"
    else:
        line += f"{'-':>11}{'-':>10}{'-':>10}{'-':>10}"
    line += f"{zb.mean():>12.3f}{zb.std():>9.3f}{zb.min():>9.2f}{zb.max():>9.2f}"
    line += f"{f'({stats[c,0]:.1f}, {stats[c,1]:.1f})':>26}"
    print(line)
print("\n(within-brain z mean~0 / std~1 confirms the z-score; note norm_stats came from the CLIPPED"
      "\n image, so inverting img->raw is exact only away from the clipped tails)")

if s["img_raw"] is not None:
    fig, ax = plt.subplots(2, len(s["names"]), figsize=(4 * len(s["names"]), 6))
    for c, nm in enumerate(s["names"]):
        ax[0, c].hist(s["img_raw"][..., c][s["mask"]], bins=120); ax[0, c].set_title(f"{nm} raw")
        ax[1, c].hist(s["img"][..., c][s["mask"]], bins=120);     ax[1, c].set_title(f"{nm} z-scored")
        ax[1, c].axvline(0, color="r", lw=1)
    fig.suptitle("Within-brain distributions: raw (top) vs z-scored (bottom, red = 0)", y=1.01)
    plt.tight_layout(); plt.show()

### Why ratios break in z-space

In [ ]:
t1_z, t1ce_z = s["img"][..., T1_IDX], s["img"][..., T1CE_IDX]
brain = s["mask"]
for thr in (0.05, 0.1, 0.25):
    frac = (np.abs(t1_z[brain]) < thr).mean()
    print(f"|T1_z| < {thr:<5}: {100 * frac:5.2f}% of brain voxels  -> ratio T1CE/T1 explodes there")
if s["img_raw"] is not None:
    t1_r = s["img_raw"][..., T1_IDX]
    print(f"\nfor comparison, raw T1 <= 0 inside brain: {100 * (t1_r[brain] <= 0).mean():.2f}% of voxels")

with np.errstate(divide="ignore", invalid="ignore"):
    q_z = (t1ce_z / t1_z)[brain]
q_z = q_z[np.isfinite(q_z)]
print(f"\nz-space ratio: median {np.median(q_z):+.2f}, but 1st/99th pct = "
      f"{np.percentile(q_z, 1):+.1f} / {np.percentile(q_z, 99):+.1f}  (heavy tails)")
if s["img_raw"] is not None:
    with np.errstate(divide="ignore", invalid="ignore"):
        q_r = (s["img_raw"][..., T1CE_IDX] / s["img_raw"][..., T1_IDX])[brain]
    q_r = q_r[np.isfinite(q_r)]
    print(f"raw ratio    : median {np.median(q_r):+.2f}, 1st/99th pct = "
          f"{np.percentile(q_r, 1):+.1f} / {np.percentile(q_r, 99):+.1f}")

# same regression, now in z-space -- shows the verdict can flip with the space
xz, yz, _ = collect(paths, space="img", region="et")
report(fit_models(xz, yz), "pooled ET voxels, z-scored (img)")

## 4. A "supervised" `S` from RAW data divided by a chosen scale

Section 3 showed why we should *not* build `S` on the z-scored `img`: the per-channel mean
subtraction destroys the additive relation (T1 and T1CE get different `mu`) and makes ratios
blow up. So instead take **`img_raw`** and divide by a single scale factor picked off the
intensity histogram.

**The divisor must be SHARED between T1 and T1CE.** With a common `s`:

    T1CE/s = T1/s + S/s      (additive survives, S just rescales)
    T1CE/s = (T1/s) * S      (multiplicative survives, S unchanged)

With different per-contrast divisors the additive form picks up a spurious gain
`(s_t1/s_t1ce)` and is no longer `x0 = x1 + S`. This is the one constraint the scale choice
has to respect.

Below: histogram the raw within-brain intensities, look at candidate percentile scales, pick
one, and rebuild `S` in that space.

In [ ]:
# ---- candidate scales from the RAW within-brain histogram, pooled over T1 and T1CE ----
# (pooled so the divisor is SHARED; see the note above)
raw_t1, raw_t1ce, per_subj_p = [], [], []
for p in paths:
    sub = load_subject(p, want_seg=False)
    if sub["img_raw"] is None:
        continue
    b = sub["mask"]
    a, c = sub["img_raw"][..., T1_IDX][b], sub["img_raw"][..., T1CE_IDX][b]
    raw_t1.append(a); raw_t1ce.append(c)
    per_subj_p.append((os.path.basename(p).replace("_img.h5", ""),
                       np.percentile(np.concatenate([a, c]), 99.0)))
assert raw_t1, "no img_raw available -- regenerate the h5 with save_raw_image: true"
raw_t1, raw_t1ce = np.concatenate(raw_t1), np.concatenate(raw_t1ce)
pooled = np.concatenate([raw_t1, raw_t1ce])

print(f"{'percentile':>12}{'T1':>12}{'T1CE':>12}{'pooled':>12}{'  max after':>13}{'  % > scale':>12}")
CANDIDATES = [95.0, 99.0, 99.5, 99.9, 100.0]
for q in CANDIDATES:
    v = np.percentile(pooled, q)
    print(f"{q:>12}{np.percentile(raw_t1, q):>12.1f}{np.percentile(raw_t1ce, q):>12.1f}"
          f"{v:>12.1f}{pooled.max() / v:>13.2f}{100 * (pooled > v).mean():>12.3f}")

# ---- pick one ----
SCALE_PCT = 99.0                      # <-- the knob; 99-99.5 keeps outliers from dominating
SCALE = float(np.percentile(pooled, SCALE_PCT))
print(f"\nSCALE = p{SCALE_PCT} of pooled raw within-brain intensity = {SCALE:.2f}")
print(f"  -> T1 in [{raw_t1.min()/SCALE:.2f}, {raw_t1.max()/SCALE:.2f}], "
      f"T1CE in [{raw_t1ce.min()/SCALE:.2f}, {raw_t1ce.max()/SCALE:.2f}] after division")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
hi = np.percentile(pooled, 99.9)
ax[0].hist(raw_t1,   bins=200, range=(0, hi), alpha=0.55, density=True, label="T1 (raw)")
ax[0].hist(raw_t1ce, bins=200, range=(0, hi), alpha=0.55, density=True, label="T1CE (raw)")
for q in CANDIDATES[:-1]:
    v = np.percentile(pooled, q)
    ax[0].axvline(v, ls="--", lw=1, color="k")
    ax[0].text(v, ax[0].get_ylim()[1] * 0.9, f"p{q:g}", rotation=90, fontsize=7, ha="right")
ax[0].axvline(SCALE, color="r", lw=2, label=f"chosen SCALE = {SCALE:.0f}")
ax[0].set_xlabel("raw intensity (within brain)"); ax[0].legend(fontsize=8)
ax[0].set_title("Raw within-brain histogram + candidate scales")

# inter-subject spread of the SAME percentile -- raw data is NOT subject-normalized
names_ = [n for n, _ in per_subj_p]; vals_ = np.array([v for _, v in per_subj_p])
ax[1].bar(range(len(vals_)), vals_)
ax[1].axhline(SCALE, color="r", lw=2)
ax[1].set_xticks(range(len(vals_))); ax[1].set_xticklabels(names_, rotation=90, fontsize=6)
ax[1].set_ylabel("p99 of raw intensity"); ax[1].set_title("Per-subject p99 (red = pooled SCALE)")
plt.tight_layout(); plt.show()
print(f"per-subject p99: min {vals_.min():.0f}  max {vals_.max():.0f}  "
      f"spread/mean {vals_.std() / vals_.mean():.2f}"
      "\n(a GLOBAL scale leaves this inter-subject variation in the data -- that is what the"
      "\n z-score was removing. SCALE_MODE below lets you use each subject's own scale instead.)")

In [ ]:
SCALE_MODE = "per_subject"     # "global" (the SCALE above) | "per_subject" (that subject's own p)

brain, et = s["mask"], s["et"]
if SCALE_MODE == "per_subject":
    b = s["mask"]
    sub_pooled = np.concatenate([s["img_raw"][..., T1_IDX][b], s["img_raw"][..., T1CE_IDX][b]])
    scale_used = float(np.percentile(sub_pooled, SCALE_PCT))
else:
    scale_used = SCALE
print(f"SCALE_MODE={SCALE_MODE}  ->  dividing BOTH T1 and T1CE by {scale_used:.2f}")

x1 = s["img_raw"][..., T1_IDX]   / scale_used      # raw / shared scale  (NOT z-scored)
x0 = s["img_raw"][..., T1CE_IDX] / scale_used

S_add = x0 - x1
with np.errstate(divide="ignore", invalid="ignore"):
    S_mul = x0 / x1
S_mul_f = np.where(np.isfinite(S_mul), S_mul, np.nan)

def summarize(S, nm):
    for region, sel in (("ET", et), ("brain", brain)):
        v = S[sel]; v = v[np.isfinite(v)]
        if v.size:
            print(f"  {nm:<7} in {region:<6}: mean {np.nanmean(v):+8.3f}  std {np.nanstd(v):7.3f}  "
                  f"p1 {np.percentile(v,1):+8.2f}  p99 {np.percentile(v,99):+8.2f}")
print(f"raw / {scale_used:.1f}  (shared divisor, so x0 = x1 + S holds exactly)")
summarize(S_add, "S_add"); summarize(S_mul_f, "S_mul")

# how much of the enhancement signal lives in ET?
print(f"\nET is {100 * et.mean():.3f}% of the FOV but carries "
      f"{100 * np.abs(S_add[et]).sum() / max(np.abs(S_add[brain]).sum(), 1e-9):.1f}% of total |S_add| mass")

zb = int(et.reshape(et.shape[0], -1).sum(1).argmax())
vmax_img = float(np.percentile(np.concatenate([x1[brain], x0[brain]]), 99.5))
sa = float(np.percentile(np.abs(S_add[brain]), 99.5))
fig, ax = plt.subplots(1, 5, figsize=(21, 4.2))
panels = [("x1 = T1_raw/%.0f" % scale_used, x1[zb], dict(cmap="gray", vmin=0, vmax=vmax_img)),
          ("x0 = T1CE_raw/%.0f" % scale_used, x0[zb], dict(cmap="gray", vmin=0, vmax=vmax_img)),
          ("S_add = x0 - x1", S_add[zb], dict(cmap="RdBu_r", vmin=-sa, vmax=sa)),
          ("S_add masked to ET", np.where(et[zb], S_add[zb], np.nan), dict(cmap="magma", vmin=0, vmax=sa)),
          ("S_mul = x0 / x1", np.clip(S_mul_f[zb], 0, 5), dict(cmap="magma", vmin=0, vmax=3))]
for a, (ttl, im, kw) in zip(ax, panels):
    h = a.imshow(im, **kw); a.set_title(ttl, fontsize=10); a.axis("off")
    plt.colorbar(h, ax=a, fraction=0.046)
    if "ET" not in ttl:
        a.contour(et[zb], levels=[0.5], colors="lime", linewidths=0.6)
fig.suptitle(f"Supervised S from raw/{scale_used:.0f} (z={zb}); green = ET outline", y=1.03)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
rng_add = (float(np.percentile(S_add[brain], 0.5)), float(np.percentile(S_add[brain], 99.5)))
ax[0].hist(S_add[brain], bins=150, range=rng_add, alpha=0.55, density=True, label="brain")
ax[0].hist(S_add[et],    bins=150, range=rng_add, alpha=0.75, density=True, label="ET")
ax[0].axvline(0, color="k", lw=0.8)
ax[0].set_title("S_add: ET vs whole brain"); ax[0].legend(); ax[0].set_xlabel("S_add")
v = S_mul_f[et]; v = v[np.isfinite(v)]
ax[1].hist(np.clip(v, 0, 10), bins=150, density=True, color="c")
ax[1].axvline(1.0, color="k", lw=0.8)
ax[1].set_title("S_mul in ET (clipped to [0,10]; 1 = no change)"); ax[1].set_xlabel("S_mul")
plt.tight_layout(); plt.show()

print(f"\nTo train in THIS space, set the loader to read raw and divide by the SAME scale:")
print(f'    "image_key": "img_raw",')
print(f'    "scales": [{scale_used:.1f}, {scale_used:.1f}, {scale_used:.1f}, {scale_used:.1f}]')
print("  (all four equal -- a shared divisor is what keeps x0 = x1 + S exact).")
print("  NOTE: only SCALE_MODE='global' is expressible this way; 'per_subject' needs a loader")
print("  change (or a per-subject scale precomputed into the h5), since `scales` is a constant.")
print(f"  Data range changes from z-scored ~[-2,3] to ~[0,{vmax_img:.1f}], so the bridge tau")
print("  (peak noise std) must be re-tuned for this space.")

## 5. Takeaways

Read the numbers above, not this cell — but the things to check:

* **Which RMSE is lower** on pooled ET voxels (section 2), and whether the same model wins
  per-subject. A split verdict means neither model is clearly right.
* **`corr(S, T1)`** — the model whose `S` is *uncorrelated* with `T1` is the one whose
  assumption actually holds.
* **The affine reference**: `a ~ 1` argues additive, `b ~ 0` argues multiplicative. If the free
  fit is much better than both, the true relationship is affine (`T1CE = a*T1 + b`) and neither
  1-parameter model captures it — worth considering as a third option.
* **Space matters**: if the verdict flips between `img_raw` and `img`, the multiplicative model
  is an artifact of the space, since ratios are meaningless once the data is mean-subtracted.
  Prefer the raw-space verdict for the *physics*.
* **Per-subject spread** of `b_add` / `a_mul` tells you whether a single global `S` is even
  plausible, or whether `S` must be predicted per-voxel.
* **The scale choice (section 4) is a real trade-off**, not a formality:
  - a *shared* divisor for T1/T1CE is mandatory — anything else breaks `x0 = x1 + S`;
  - a **global** scale is what `scales` in the config can express today, but it leaves the
    inter-subject intensity variation that z-scoring used to remove (see the per-subject p99 bar
    chart — if that spread is large, expect the network to see quite different input ranges
    per subject);
  - a **per-subject** shared scale fixes that and still preserves additivity, but needs a loader
    change since `scales` is a constant.